# V8 · Stage 1.2d — Frozen hybrid selector (minimum valid)

**Scope**: minimum-valid version for the 2026-07-15 submission (see memory
`v8-scope-frozen-july15`).  No LOAO/LOSO, no full bootstrap procedure, no
context-length sweep, no abstention grey zone.  One predeclared threshold
method, evaluated once on the three external cells.

**Depends on**: v8 clean checkpoint (`pinn_phase3_v8_clean.pt`).

## Method (predeclared BEFORE seeing external cells)

1. **Canonical residual — context holdout residual**: fit exp `SoH = a·exp(-b·n) + c` on the first 80 % of the 50-cycle context (cycles 0..39); RMSE (pp) on the last 20 % (cycles 40..49) predicted from that fit.
2. **Split roles** — carve `validation_selector` (~1,026 rows) and `final_test` (~1,026 rows) sim-disjoint from the v8 corpus test split.  Anchor-stratified 50/50 by sim.
3. **Threshold selection** — sweep τ across a fine grid; pick τ* minimising **sim-weighted mean hybrid RMSE** on the `validation_selector` pool.  Sim-weighted (not row-weighted) so long trajectories with many windows do not dominate.
4. **Freeze τ**.  Report a single τ_final = τ*_L1 (Level-1 only; no Level-2 recalibration in this scope).
5. **Evaluate frozen τ once** on the three external cells (CALB_0029, EVE_0003, REPT_0028).  Report the minimum-valid table:

| Strategy | Mean RMSE |
|---|---|
| Exponential | ... |
| v8 operator | ... |
| Frozen hybrid | ... |
| Oracle | ... |

Plus per-cell selector decisions.

## Expected outputs
- `outputs/results/hybrid_frozen_summary.md`
- `outputs/results/hybrid_frozen_table.parquet`
- `outputs/results/hybrid_frozen_perbin_decisions.json`


## 1. Setup

In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy.optimize import curve_fit

PROJ = Path("/home/hj/Desktop/PINNs")
sys.path.insert(0, str(PROJ / "Voltaris" / "Data_Exploration"))
from phase3_v7_validate import OperatorV7, forecast_v7
from phase3_validate import _load_longterm_soh

V8_CKPT = PROJ / "outputs" / "models" / "pinn_phase3_v8_clean.pt"
V8_DATASET = PROJ / "configs" / "phase3_corpus" / "_v8_dataset.parquet"
assert V8_CKPT.exists(), "01_1c v8 clean checkpoint required"

SEED = 456; K = 50; CTX_TAIL = 10
rng = np.random.default_rng(SEED); np.random.seed(SEED)


## 2. Canonical residual helper

In [2]:
def _exp(n, a, b, c): return a * np.exp(-b * n) + c

def context_holdout_residual(context_soh, K=K, tail=CTX_TAIL):
    n_fit = np.arange(K - tail, dtype=float)
    y_fit = np.asarray(context_soh[:K - tail], dtype=float)
    n_val = np.arange(K - tail, K, dtype=float)
    y_val = np.asarray(context_soh[K - tail:K], dtype=float)
    popt, _ = curve_fit(_exp, n_fit, y_fit,
                        p0=[y_fit[0]-y_fit[-1], 1e-3, y_fit[-1]], maxfev=10000)
    y_hat = _exp(n_val, *popt)
    return float(np.sqrt(np.mean((y_hat - y_val)**2)) * 100), popt

def rmse_pp(y_hat, y_true):
    return float(np.sqrt(np.mean((np.asarray(y_hat) - np.asarray(y_true))**2)) * 100)


## 3. Carve `validation_selector` and `final_test` from the corpus test split

In [3]:
df = pd.read_parquet(V8_DATASET)
test_rows = df[df["split"] == "test"].copy()
sims_test = (test_rows[["anchor_id","sample_id"]]
             .drop_duplicates().reset_index(drop=True))
role = np.empty(len(sims_test), dtype=object)
for aid in sims_test["anchor_id"].unique():
    idx = np.array(sims_test.index[sims_test["anchor_id"] == aid])
    rng.shuffle(idx)
    half = len(idx) // 2
    role[idx[:half]] = "validation_selector"
    role[idx[half:]] = "final_test"
sims_test["role"] = role
test_rows = test_rows.merge(sims_test, on=["anchor_id","sample_id"])
print(test_rows.groupby(["anchor_id","role"]).size().unstack(fill_value=0))


role       final_test  validation_selector
anchor_id                                 
CALB_0003         160                  160
CALB_0009         152                  152
CALB_0010         144                  160
CALB_0015         160                  160
EVE_0004          160                  160
REPT_0007         136                  104
REPT_0057         160                   84


## 4. Score every synthetic row with residual + exp + operator RMSE

In [4]:
# Load v8 operator
ckpt = torch.load(V8_CKPT, map_location="cpu", weights_only=False)
model = OperatorV7(K=K)
model.load_state_dict(ckpt["state_dict"])
model.set_x_health_stats(torch.tensor(ckpt["xh_mean"]), torch.tensor(ckpt["xh_std"]))
model.set_theta_stats(torch.tensor(ckpt["th_mean"]), torch.tensor(ckpt["th_std"]))
model.eval()

def _op_forecast(row, target_cycles):
    with torch.no_grad():
        pred = model(
            torch.tensor(np.asarray(row["x_health"], dtype=np.float32)).unsqueeze(0),
            torch.tensor(np.asarray(row["theta_norm"], dtype=np.float32)).unsqueeze(0),
            torch.tensor(np.asarray(row["context_delta"], dtype=np.float32)).unsqueeze(0),
            torch.tensor([float(row["context_soh_start"])]),
            torch.tensor(np.asarray(target_cycles, dtype=np.float32)),
        ).squeeze(0).cpu().numpy()
    return pred

records = []
for _, row in test_rows.iterrows():
    context_soh = float(row["context_soh_start"]) + np.asarray(row["context_delta"], dtype=np.float64)
    resid, _ = context_holdout_residual(context_soh, K=K, tail=CTX_TAIL)
    tgt_cycles = np.asarray(row["target_cycles"], dtype=np.int64)
    tgt_soh = np.asarray(row["target_soh"], dtype=np.float64)
    if not np.isfinite(resid) or len(tgt_cycles) < 5:
        continue
    try:
        popt, _ = curve_fit(_exp, np.arange(K, dtype=float), context_soh,
                            p0=[context_soh[0]-context_soh[-1], 1e-3, context_soh[-1]],
                            maxfev=10000)
        exp_pred = _exp(tgt_cycles.astype(float), *popt)
        rmse_exp = rmse_pp(exp_pred, tgt_soh)
    except Exception:
        continue
    op_pred = _op_forecast(row, tgt_cycles)
    rmse_op = rmse_pp(op_pred, tgt_soh)
    records.append({
        "anchor_id": row["anchor_id"], "sample_id": row["sample_id"],
        "role": row["role"],
        "residual_pp": resid,
        "rmse_exp_pp": rmse_exp,
        "rmse_op_pp": rmse_op,
        "oracle_choice": "exp" if rmse_exp <= rmse_op else "op",
        "oracle_rmse_pp": min(rmse_exp, rmse_op),
    })
synth = pd.DataFrame(records)
print(f"scored rows: {len(synth):,}")
print(synth.groupby("role").size())


/tmp/ipykernel_40713/3506200811.py:8: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(_exp, n_fit, y_fit,
/tmp/ipykernel_40713/3522673077.py:29: OptimizeWarning: Covariance of the parameters could not be estimated
  popt, _ = curve_fit(_exp, np.arange(K, dtype=float), context_soh,


scored rows: 2,052
role
final_test             1072
validation_selector     980
dtype: int64


## 5. Threshold selection on `validation_selector`

Sim-weighted mean is authoritative — every simulated trajectory contributes
equally, independent of how many windows it produced.

In [5]:
def sim_weighted_mean(df, values):
    tmp = df.copy(); tmp["_v"] = values
    return float(tmp.groupby(["anchor_id","sample_id"])["_v"].mean().mean())

pool = synth[synth["role"] == "validation_selector"].copy()

taus = np.linspace(0.01, 3.0, 300)
def hybrid_at(pool, tau):
    return np.where(pool["residual_pp"].values < tau,
                    pool["rmse_exp_pp"].values,
                    pool["rmse_op_pp"].values)

sim_means = [sim_weighted_mean(pool, hybrid_at(pool, t)) for t in taus]
tau_star = float(taus[int(np.argmin(sim_means))])
best_sim_mean = float(np.min(sim_means))
print(f"τ* (sim-weighted mean minimiser) = {tau_star:.3f} pp | best mean = {best_sim_mean:.4f} pp")


τ* (sim-weighted mean minimiser) = 0.540 pp | best mean = 4.7539 pp


## 6. Freeze τ. Evaluate ONCE on the three external cells.

In [6]:
EXTERNAL = [("0029","CALB"), ("0003","EVE"), ("0028","REPT")]
ext = []
for cid, mk in EXTERNAL:
    obs_n, obs_soh = _load_longterm_soh(cid, mk)
    ctx_soh = obs_soh[:K]
    resid, _ = context_holdout_residual(ctx_soh, K=K, tail=CTX_TAIL)
    tgt_n = obs_n[K:]; tgt_soh = obs_soh[K:]
    popt, _ = curve_fit(_exp, np.arange(K, dtype=float), ctx_soh,
                        p0=[ctx_soh[0]-ctx_soh[-1], 1e-3, ctx_soh[-1]], maxfev=10000)
    exp_pred = _exp(tgt_n.astype(float), *popt)
    rmse_exp = rmse_pp(exp_pred, tgt_soh)
    r_op = forecast_v7(model, cid, mk, K=K, forecast_len=int(len(tgt_n)))
    rmse_op = float(r_op["rmse_pp_covered"])
    picked = "exp" if resid < tau_star else "op"
    ext.append({
        "cell": f"{mk}_{cid}",
        "residual_pp": resid,
        "rmse_exp_pp": rmse_exp,
        "rmse_op_pp": rmse_op,
        "oracle_pick": "exp" if rmse_exp <= rmse_op else "op",
        "hybrid_pick": picked,
        "hybrid_rmse_pp": rmse_exp if picked == "exp" else rmse_op,
        "oracle_rmse_pp": min(rmse_exp, rmse_op),
    })
ext_df = pd.DataFrame(ext)
ext_df


,cell,residual_pp,rmse_exp_pp,rmse_op_pp,oracle_pick,hybrid_pick,hybrid_rmse_pp,oracle_rmse_pp
0,CALB_0029,0.273295,0.431714,1.059375,exp,exp,0.431714,0.431714
1,EVE_0003,0.707057,1.545855,0.840880,op,op,0.840880,0.840880
2,REPT_0028,0.100560,0.131690,0.511496,exp,exp,0.131690,0.131690


## 7. Minimum-valid table + per-cell decisions

In [7]:
summary = pd.DataFrame([
    {"strategy":"Exponential",   "mean_rmse_pp": float(ext_df["rmse_exp_pp"].mean())},
    {"strategy":"v8 operator",   "mean_rmse_pp": float(ext_df["rmse_op_pp"].mean())},
    {"strategy":"Frozen hybrid", "mean_rmse_pp": float(ext_df["hybrid_rmse_pp"].mean())},
    {"strategy":"Oracle",        "mean_rmse_pp": float(ext_df["oracle_rmse_pp"].mean())},
])
summary


,strategy,mean_rmse_pp
0,Exponential,0.703086
1,v8 operator,0.803917
2,Frozen hybrid,0.468095
3,Oracle,0.468095


In [8]:
summary.to_parquet(PROJ / "outputs/results/hybrid_frozen_table.parquet", index=False)
per_cell = {r["cell"]: {"residual_pp": r["residual_pp"],
                        "hybrid_pick": r["hybrid_pick"],
                        "oracle_pick": r["oracle_pick"]}
            for _, r in ext_df.iterrows()}
(PROJ / "outputs/results/hybrid_frozen_perbin_decisions.json").write_text(
    json.dumps({"tau_star_pp": tau_star, "per_cell": per_cell}, indent=2))

md = ["# Frozen hybrid selector — minimum valid result",
      "",
      f"τ* = **{tau_star:.3f} pp**  (chosen on {len(pool):,}-row selector pool, sim-weighted mean minimiser)",
      "",
      "## Mean RMSE across the three external cells",
      "",
      summary.to_markdown(index=False, floatfmt=".3f"),
      "",
      "## Per-cell decisions",
      "",
      ext_df.to_markdown(index=False, floatfmt=".3f")]
report = "\n".join(md)
(PROJ / "outputs/results/hybrid_frozen_summary.md").write_text(report)
print(report)


# Frozen hybrid selector — minimum valid result

τ* = **0.540 pp**  (chosen on 980-row selector pool, sim-weighted mean minimiser)

## Mean RMSE across the three external cells

| strategy      |   mean_rmse_pp |
|:--------------|---------------:|
| Exponential   |          0.703 |
| v8 operator   |          0.804 |
| Frozen hybrid |          0.468 |
| Oracle        |          0.468 |

## Per-cell decisions

| cell      |   residual_pp |   rmse_exp_pp |   rmse_op_pp | oracle_pick   | hybrid_pick   |   hybrid_rmse_pp |   oracle_rmse_pp |
|:----------|--------------:|--------------:|-------------:|:--------------|:--------------|-----------------:|-----------------:|
| CALB_0029 |         0.273 |         0.432 |        1.059 | exp           | exp           |            0.432 |            0.432 |
| EVE_0003  |         0.707 |         1.546 |        0.841 | op            | op            |            0.841 |            0.841 |
| REPT_0028 |         0.101 |         0.132 |        0.511 | exp

## 8. Verdict marker

Fill in after execution:

- [ ] **HYBRID FRAMING WINS** — frozen hybrid mean RMSE < both fixed strategies AND
      sensible decisions on ≥ 2 of 3 cells (agrees with oracle).
- [ ] **OPERATOR FRAMING WINS** — v8 operator beats exp on ≥ 2 of 3 external cells.
- [ ] **PRELIMINARY COMPARISON FRAMING** — neither wins; contribution reframed as
      *"we compare empirical extrapolation and a PyBaMM-trained neural operator
      and find complementary performance across degradation profiles"*.

This verdict feeds directly into tomorrow morning's abstract-framing decision
(memory `v8-scope-frozen-july15`).
